Yêu cầu 1: Sử dụng Mô hình Markov Ẩn (HMM) để xác định các chế độ thị trường tiềm ẩn (ví dụ: tăng giá, giảm giá, biến động cao/thấp) trong 20 năm cho SPY và 2 cổ phiếu khác (BAC, AES, DCOM).
Yêu cầu 2: Phân tích cách độ nhạy yếu tố của từng tài sản thay đổi qua các chế độ.
Yêu cầu cụ thể:
Sử dụng HMM để xác định 3 trạng thái (chế độ) cho SPY, BAC, AES, DCOM.
Sử dụng Mô hình Đa yếu tố Fama để tìm mức độ phơi nhiễm yếu tố của các cổ phiếu trong chế độ riêng của chúng.
Sử dụng Mô hình Đa yếu tố Fama để tìm mức độ phơi nhiễm yếu tố của các cổ phiếu dưới chế độ thị trường (SPY).

In [7]:
# Cài đặt các thư viện cần thiết trong Google Colab
!pip install yfinance hmmlearn pandas_datareader statsmodels

# Nhập các thư viện cần thiết
import yfinance as yf
import pandas as pd
import numpy as np
from hmmlearn.hmm import GaussianHMM
import pandas_datareader.data as web
import statsmodels.api as sm

# Bước 1: Thu thập dữ liệu giá lịch sử cho SPY, BAC, AES, DCOM
tickers = ['SPY', 'BAC', 'AES', 'DCOM']
start_date = '2005-01-01'
end_date = '2025-05-31'  # Giới hạn đến cuối tháng trước để khớp với dữ liệu Fama-French

# Tải dữ liệu giá hàng tháng
data_monthly = yf.download(tickers, start=start_date, end=end_date, interval='1mo')

# Kiểm tra cấu trúc dữ liệu
print("Cột có trong dữ liệu giá:", data_monthly.columns)

# Chọn cột 'Close' từ MultiIndex
returns_monthly = data_monthly['Close'].pct_change().dropna()

# Chuyển chỉ số thành định dạng PeriodIndex hàng tháng
returns_monthly.index = returns_monthly.index.to_period('M')
print("Dữ liệu lợi nhuận hàng tháng:\n", returns_monthly.head())
print("Phạm vi chỉ số của returns_monthly:", returns_monthly.index)

# Bước 2: Sử dụng HMM để xác định 3 trạng thái (chế độ) cho từng tài sản
regimes = {}
for ticker in tickers:
    asset_returns = returns_monthly[ticker].values.reshape(-1, 1)
    hmm_model = GaussianHMM(n_components=3, covariance_type="full", n_iter=1000)
    hmm_model.fit(asset_returns)
    states = hmm_model.predict(asset_returns)
    regimes[ticker] = pd.Series(states, index=returns_monthly.index)
    print(f"Trạng thái chế độ cho {ticker} (0, 1, 2):\n", regimes[ticker].head())

# Bước 3: Tải dữ liệu yếu tố Fama-French bốn yếu tố
try:
    # Tải dữ liệu 3 yếu tố cơ bản
    ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2005-01-01', end='2025-05-31')[0]

    # Tải dữ liệu yếu tố momentum riêng
    ff_mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='2005-01-01', end='2025-05-31')[0]

    # Kết hợp dữ liệu
    ff4 = ff3.join(ff_mom['Mom'])

    # Kiểm tra loại chỉ số
    print("Loại chỉ số của ff4:", type(ff4.index))
    print("Cột trong dữ liệu Fama-French:", ff4.columns)
    print("Phạm vi chỉ số của ff4:", ff4.index)
except Exception as e:
    print(f"Lỗi khi tải dữ liệu Fama-French: {e}")
    raise

# Bước 4: Căn chỉnh dữ liệu và tính lợi nhuận dư
# Đảm bảo ff4.index là PeriodIndex
if isinstance(ff4.index, pd.DatetimeIndex):
    ff4.index = ff4.index.to_period('M')
common_index = returns_monthly.index.intersection(ff4.index)
print("Chỉ số chung sau khi giao nhau:", common_index)

if len(common_index) == 0:
    raise ValueError("Không có giao nhau giữa chỉ số của returns_monthly và ff4. Kiểm tra lại ngày bắt đầu và kết thúc.")

returns_monthly = returns_monthly.loc[common_index]
ff4 = ff4.loc[common_index]

# Chuyển đổi yếu tố Fama-French từ phần trăm sang thập phân (chia cho 100)
ff4[['Mkt-RF', 'SMB', 'HML', 'Mom', 'RF']] = ff4[['Mkt-RF', 'SMB', 'HML', 'Mom', 'RF']] / 100

excess_returns = returns_monthly.sub(ff4['RF'], axis=0)
print("Dữ liệu lợi nhuận dư:\n", excess_returns.head())
factors = ff4[['Mkt-RF', 'SMB', 'HML', 'Mom']]

# Bước 5: Sử dụng Fama Multifactor Model để tìm mức độ phơi nhiễm yếu tố trong chế độ riêng
for ticker in tickers:
    print(f"\nPhân tích độ nhạy yếu tố cho {ticker} dưới các chế độ riêng:")
    asset_regimes = regimes[ticker].loc[common_index]
    for k in range(3):
        mask = (asset_regimes == k)
        X = sm.add_constant(factors.loc[mask])
        y = excess_returns[ticker].loc[mask]
        if len(y) > 0:
            model = sm.OLS(y, X).fit()
            print(f"\n{ticker} Chế độ {k} Mức độ phơi nhiễm yếu tố (Beta):")
            print(model.params)
        else:
            print(f"\n{ticker} Chế độ {k} Không có đủ dữ liệu để phân tích.")

# Bước 6: Sử dụng Fama Multifactor Model để tìm mức độ phơi nhiễm yếu tố dưới chế độ thị trường (SPY)
market_regimes = regimes['SPY'].loc[common_index]
print("\nPhân tích độ nhạy yếu tố cho BAC, AES, DCOM dưới các chế độ thị trường (SPY):")
for k in range(3):
    mask = (market_regimes == k)
    X = sm.add_constant(factors.loc[mask])
    for ticker in ['BAC', 'AES', 'DCOM']:
        y = excess_returns[ticker].loc[mask]
        if len(y) > 0:
            model = sm.OLS(y, X).fit()
            print(f"\n{ticker} dưới Chế độ SPY {k} Mức độ phơi nhiễm yếu tố (Beta):")
            print(model.params)
        else:
            print(f"\n{ticker} dưới Chế độ SPY {k} Không có đủ dữ liệu để phân tích.")

/tmp/ipython-input-7-4065435661.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_monthly = yf.download(tickers, start=start_date, end=end_date, interval='1mo')
[*********************100%***********************]  4 of 4 completed


Cột có trong dữ liệu giá: MultiIndex([( 'Close',  'AES'),
            ( 'Close',  'BAC'),
            ( 'Close', 'DCOM'),
            ( 'Close',  'SPY'),
            (  'High',  'AES'),
            (  'High',  'BAC'),
            (  'High', 'DCOM'),
            (  'High',  'SPY'),
            (   'Low',  'AES'),
            (   'Low',  'BAC'),
            (   'Low', 'DCOM'),
            (   'Low',  'SPY'),
            (  'Open',  'AES'),
            (  'Open',  'BAC'),
            (  'Open', 'DCOM'),
            (  'Open',  'SPY'),
            ('Volume',  'AES'),
            ('Volume',  'BAC'),
            ('Volume', 'DCOM'),
            ('Volume',  'SPY')],
           names=['Price', 'Ticker'])
Dữ liệu lợi nhuận hàng tháng:
 Ticker        AES       BAC      DCOM       SPY
Date                                           
2005-02  0.191459  0.006038  0.016394  0.020904
2005-03 -0.021505 -0.054662  0.003225 -0.022134
2005-04 -0.018315  0.031171 -0.046499 -0.014881
2005-05 -0.074005  0.028

/tmp/ipython-input-7-4065435661.py:44: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2005-01-01', end='2025-05-31')[0]
/tmp/ipython-input-7-4065435661.py:44: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2005-01-01', end='2025-05-31')[0]


Loại chỉ số của ff4: <class 'pandas.core.indexes.period.PeriodIndex'>
Cột trong dữ liệu Fama-French: Index(['Mkt-RF', 'SMB', 'HML', 'RF', 'Mom'], dtype='object')
Phạm vi chỉ số của ff4: PeriodIndex(['2005-01', '2005-02', '2005-03', '2005-04', '2005-05', '2005-06',
             '2005-07', '2005-08', '2005-09', '2005-10',
             ...
             '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
             '2025-01', '2025-02', '2025-03', '2025-04'],
            dtype='period[M]', name='Date', length=244)
Chỉ số chung sau khi giao nhau: PeriodIndex(['2005-02', '2005-03', '2005-04', '2005-05', '2005-06', '2005-07',
             '2005-08', '2005-09', '2005-10', '2005-11',
             ...
             '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12',
             '2025-01', '2025-02', '2025-03', '2025-04'],
            dtype='period[M]', name='Date', length=243)
Dữ liệu lợi nhuận dư:
 Ticker        AES       BAC      DCOM       SPY
Date               

/tmp/ipython-input-7-4065435661.py:47: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='2005-01-01', end='2025-05-31')[0]
/tmp/ipython-input-7-4065435661.py:47: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='2005-01-01', end='2025-05-31')[0]
